# NBA Data Preparation

Assemblage du DataFrame ML final à partir des fichiers scrappés.

**Cibles :**
- `home_win` : 1 si l'équipe domicile gagne, 0 sinon (classification)
- `point_diff` : home_score - away_score (régression)

**Pipeline :**
1. Charger les données
2. Joindre les stats avancées par match
3. Calculer les features de fatigue (back-to-back, jours de repos)
4. Calculer les features de forme récente (rolling N matchs)
5. Joindre les stats joueurs (absences proxy)
6. Sauvegarder le DataFrame ML final

In [3]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

print('Imports OK')

Imports OK


## 1. Chargement des données

In [4]:
games      = pd.read_csv('data/nba_games.csv')
adv        = pd.read_csv('data/nba_boxscore_advanced.csv')
players    = pd.read_csv('data/nba_player_stats.csv')
teams      = pd.read_csv('data/nba_teams.csv')
games_2526 = pd.read_csv('data/nba_games_2526.csv')
team_stats_current = pd.read_csv('data/nba_team_stats_current.csv')
team_stats_recent  = pd.read_csv('data/nba_team_stats_recent.csv')
rosters    = pd.read_csv('data/nba_rosters_current.csv')
player_current = pd.read_csv('data/nba_player_stats_current.csv')

print(f'games      : {games.shape}')
print(f'adv        : {adv.shape}')
print(f'players    : {players.shape}')
print(f'games_2526 : {games_2526.shape}')
print(f'team_stats_current : {team_stats_current.shape}')
print(f'rosters    : {rosters.shape}')

games      : (3935, 63)
adv        : (7870, 30)
players    : (62149, 30)
games_2526 : (1225, 63)
team_stats_current : (30, 46)
rosters    : (530, 18)


In [5]:
# Nettoyage de base
games['date'] = pd.to_datetime(games['date'])
games = games.sort_values('date').reset_index(drop=True)

# Vérifier les cibles
print('Distribution home_win :')
print(games['home_win'].value_counts())
print(f'\nTaux victoire domicile : {games["home_win"].mean():.1%}')
print(f'Point diff moyen       : {games["point_diff"].mean():.2f}')
print(f'Point diff std         : {games["point_diff"].std():.2f}')

Distribution home_win :
home_win
1    2195
0    1740
Name: count, dtype: int64

Taux victoire domicile : 55.8%
Point diff moyen       : 2.27
Point diff std         : 15.13


## 2. Jointure stats avancées par match

Joindre `nba_boxscore_advanced.csv` sur `GAME_ID + TEAM_ID`  
→ On obtient les stats avancées de chaque équipe pour chaque match joué.

In [6]:
# Colonnes utiles des stats avancées
ADV_COLS = [
    'gameId', 'teamId', 'teamTricode',
    'estimatedOffensiveRating', 'estimatedDefensiveRating', 'estimatedNetRating',
    'estimatedPace', 'assistRatio',
    'offensiveReboundPercentage', 'defensiveReboundPercentage',
    'effectiveFieldGoalPercentage', 'trueShootingPercentage', 'PIE'
]

adv_clean = adv[[c for c in ADV_COLS if c in adv.columns]].copy()
adv_clean = adv_clean.rename(columns={'gameId': 'GAME_ID', 'teamId': 'TEAM_ID'})

print(f'Stats avancées : {adv_clean.shape}')
adv_clean.head(2)

Stats avancées : (7870, 13)


,GAME_ID,TEAM_ID,teamTricode,estimatedOffensiveRating,estimatedDefensiveRating,estimatedNetRating,estimatedPace,assistRatio,offensiveReboundPercentage,defensiveReboundPercentage,effectiveFieldGoalPercentage,trueShootingPercentage,PIE
0,22200001,1610612738,BOS,649.500,596.900,52.500,19.500,18.500,0.256,0.810,0.634,0.668,0.566
1,22200001,1610612755,PHI,596.900,649.500,-52.500,19.500,13.100,0.190,0.744,0.581,0.634,0.434


In [7]:
# Joindre pour l'équipe domicile
adv_home = adv_clean.add_prefix('adv_home_')
adv_home = adv_home.rename(columns={
    'adv_home_GAME_ID': 'GAME_ID',
    'adv_home_TEAM_ID': 'home_TEAM_ID'
})

# Joindre pour l'équipe extérieure
adv_away = adv_clean.add_prefix('adv_away_')
adv_away = adv_away.rename(columns={
    'adv_away_GAME_ID': 'GAME_ID',
    'adv_away_TEAM_ID': 'away_TEAM_ID'
})

df = games.copy()
df = df.merge(adv_home, on=['GAME_ID', 'home_TEAM_ID'], how='left')
df = df.merge(adv_away, on=['GAME_ID', 'away_TEAM_ID'], how='left')

print(f'Après jointure stats avancées : {df.shape}')
print(f'Couverture stats avancées : {df["adv_home_estimatedNetRating"].notna().mean():.1%}')

Après jointure stats avancées : (3935, 85)
Couverture stats avancées : 100.0%


In [8]:
# Feature clé : différentiel Net Rating
df['net_rating_diff'] = df['adv_home_estimatedNetRating'] - df['adv_away_estimatedNetRating']
df['pace_diff']       = df['adv_home_estimatedPace']      - df['adv_away_estimatedPace']
df['efg_diff']        = df['adv_home_effectiveFieldGoalPercentage'] - df['adv_away_effectiveFieldGoalPercentage']

print('Corrélation Net Rating diff → home_win :')
print(df[['net_rating_diff', 'home_win', 'point_diff']].corr())

Corrélation Net Rating diff → home_win :
                 net_rating_diff  home_win  point_diff
net_rating_diff            1.000     0.792       0.996
home_win                   0.792     1.000       0.795
point_diff                 0.996     0.795       1.000


## 3. Features fatigue — Back-to-back & jours de repos

In [9]:
def compute_rest_features(df):
    """
    Calcule les jours de repos et back-to-back pour chaque équipe.
    Un back-to-back = match la veille (0 jours de repos).
    """
    records = []
    for _, row in df.iterrows():
        records.append({'GAME_ID': row['GAME_ID'], 'date': row['date'],
                        'team': row['home_TEAM_ABBREVIATION'], 'side': 'home'})
        records.append({'GAME_ID': row['GAME_ID'], 'date': row['date'],
                        'team': row['away_TEAM_ABBREVIATION'], 'side': 'away'})

    hist = pd.DataFrame(records).sort_values(['team', 'date'])
    hist['prev_date'] = hist.groupby('team')['date'].shift(1)
    hist['rest_days'] = (hist['date'] - hist['prev_date']).dt.days - 1
    hist['rest_days'] = hist['rest_days'].fillna(7).clip(0, 14)
    hist['is_b2b']    = (hist['rest_days'] == 0).astype(int)

    home_rest = hist[hist['side'] == 'home'][['GAME_ID', 'rest_days', 'is_b2b']].copy()
    home_rest.columns = ['GAME_ID', 'home_rest_days', 'home_is_b2b']
    away_rest = hist[hist['side'] == 'away'][['GAME_ID', 'rest_days', 'is_b2b']].copy()
    away_rest.columns = ['GAME_ID', 'away_rest_days', 'away_is_b2b']

    df = df.merge(home_rest, on='GAME_ID').merge(away_rest, on='GAME_ID')
    df['rest_diff'] = df['home_rest_days'] - df['away_rest_days']
    df['b2b_advantage'] = df['away_is_b2b'] - df['home_is_b2b']  # positif = avantage home
    return df

df = compute_rest_features(df)

print('Impact back-to-back sur home_win :')
print(df.groupby(['home_is_b2b', 'away_is_b2b'])['home_win'].agg(['mean', 'count']).round(3))

Impact back-to-back sur home_win :
                         mean  count
home_is_b2b away_is_b2b             
0           0           0.557   2845
            1           0.628    503
1           0           0.473    404
            1           0.557    183


## 4. Features forme récente — Rolling window

Stats moyennes sur les N derniers matchs avant chaque match.  
**Équivalent football :** xG ratio sur 5 derniers matchs.

In [10]:
def compute_rolling_features(df, n=10):
    """
    Calcule les moyennes glissantes sur les N derniers matchs
    pour chaque équipe : win%, point_diff moyen, pts marqués/concédés.
    """
    records = []
    for _, row in df.iterrows():
        # Vue domicile
        records.append({
            'GAME_ID': row['GAME_ID'], 'date': row['date'],
            'team': row['home_TEAM_ABBREVIATION'],
            'side': 'home',
            'won': row['home_win'],
            'pts_for': row['home_PTS'],
            'pts_against': row['away_PTS'],
            'pt_diff': row['point_diff'],
        })
        # Vue extérieur
        records.append({
            'GAME_ID': row['GAME_ID'], 'date': row['date'],
            'team': row['away_TEAM_ABBREVIATION'],
            'side': 'away',
            'won': 1 - row['home_win'],
            'pts_for': row['away_PTS'],
            'pts_against': row['home_PTS'],
            'pt_diff': -row['point_diff'],
        })

    hist = pd.DataFrame(records).sort_values(['team', 'date'])

    # Rolling sur N matchs précédents (shift pour éviter le data leakage)
    for col in ['won', 'pts_for', 'pts_against', 'pt_diff']:
        hist[f'roll_{col}_{n}'] = (
            hist.groupby('team')[col]
            .transform(lambda x: x.shift(1).rolling(n, min_periods=3).mean())
        )

    # Séparer home et away
    home_roll = hist[hist['side'] == 'home'][[
        'GAME_ID',
        f'roll_won_{n}', f'roll_pts_for_{n}',
        f'roll_pts_against_{n}', f'roll_pt_diff_{n}'
    ]].copy()
    home_roll.columns = ['GAME_ID'] + [f'home_{c}' for c in home_roll.columns[1:]]

    away_roll = hist[hist['side'] == 'away'][[
        'GAME_ID',
        f'roll_won_{n}', f'roll_pts_for_{n}',
        f'roll_pts_against_{n}', f'roll_pt_diff_{n}'
    ]].copy()
    away_roll.columns = ['GAME_ID'] + [f'away_{c}' for c in away_roll.columns[1:]]

    df = df.merge(home_roll, on='GAME_ID', how='left')
    df = df.merge(away_roll, on='GAME_ID', how='left')

    # Différentiels
    df[f'win_pct_diff_{n}'] = df[f'home_roll_won_{n}'] - df[f'away_roll_won_{n}']
    df[f'pt_diff_diff_{n}'] = df[f'home_roll_pt_diff_{n}'] - df[f'away_roll_pt_diff_{n}']

    return df

df = compute_rolling_features(df, n=10)

print('Aperçu features rolling :')
roll_cols = [c for c in df.columns if 'roll_' in c]
print(df[['GAME_ID', 'home_win'] + roll_cols[:6]].head(5))

Aperçu features rolling :
    GAME_ID  home_win  home_roll_won_10  home_roll_pts_for_10  \
0  22200001         1               NaN                   NaN   
1  22200002         1               NaN                   NaN   
2  22200005         1               NaN                   NaN   
3  22200007         0               NaN                   NaN   
4  22200014         0               NaN                   NaN   

   home_roll_pts_against_10  home_roll_pt_diff_10  away_roll_won_10  \
0                       NaN                   NaN               NaN   
1                       NaN                   NaN               NaN   
2                       NaN                   NaN               NaN   
3                       NaN                   NaN               NaN   
4                       NaN                   NaN               NaN   

   away_roll_pts_for_10  
0                   NaN  
1                   NaN  
2                   NaN  
3                   NaN  
4                   NaN  


In [11]:
# Corrélations des features rolling avec la cible
print('Corrélations features → home_win :')
corr_cols = [c for c in df.columns if 'roll_' in c or 'diff' in c]
corrs = df[corr_cols + ['home_win']].corr()['home_win'].drop('home_win')
print(corrs.sort_values(ascending=False).to_string())

Corrélations features → home_win :
point_diff                  0.795
net_rating_diff             0.792
efg_diff                    0.660
pt_diff_diff_10             0.292
win_pct_diff_10             0.258
home_roll_pt_diff_10        0.229
home_roll_won_10            0.202
home_roll_pts_for_10        0.134
away_roll_pts_against_10    0.112
rest_diff                   0.049
away_roll_pts_for_10       -0.107
home_roll_pts_against_10   -0.132
away_roll_won_10           -0.163
away_roll_pt_diff_10       -0.187
pace_diff                     NaN


## 5. Features joueurs — Proxy absences

Un joueur absent d'un match alors que son équipe jouait = indisponible.  
On calcule le % de minutes "habituelles" absent pour chaque équipe.

In [12]:
# Vérifier les colonnes disponibles dans player_stats
print('Colonnes player_stats :')
print(players.columns.tolist())

Colonnes player_stats :
['SEASON_ID', 'Player_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE', 'player_id', 'player_name', 'season']


In [14]:
print(players.columns.tolist())

['SEASON_ID', 'Player_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE', 'player_id', 'player_name', 'season']


In [15]:
if 'MIN' in players.columns and 'Game_ID' in players.columns:
    # Extraire l'équipe depuis MATCHUP
    players['team'] = players['MATCHUP'].str.split(' ').str[0]

    # Minutes moyennes par joueur par saison
    avg_min = (
        players.groupby(['player_id', 'season'])['MIN']
        .mean()
        .reset_index()
        .rename(columns={'MIN': 'avg_min'})
    )

    # Joueurs importants : >20 min/game
    key_players = avg_min[avg_min['avg_min'] >= 20]
    print(f'Joueurs clés (>20 min/game) : {len(key_players)}')

    # Nombre de joueurs clés disponibles par équipe par match
    player_game = players.merge(key_players, on=['player_id', 'season'], how='inner')
    games_played = player_game.groupby(['Game_ID', 'team'])['player_id'].count().reset_index()
    games_played.columns = ['GAME_ID', 'team', 'key_players_available']
    print('\nAperçu joueurs disponibles par match :')
    print(games_played.head())
else:
    print('Colonnes manquantes')
    print(players.columns.tolist())

Joueurs clés (>20 min/game) : 685

Aperçu joueurs disponibles par match :
    GAME_ID team  key_players_available
0  22200001  BOS                      6
1  22200001  PHI                      5
2  22200002  GSW                      8
3  22200002  LAL                      4
4  22200003  DET                      5


## 6. Nettoyage final & sélection des features ML

In [16]:
# Features ML finales
FEATURE_COLS = [
    # Contexte
    'GAME_ID', 'date', 'home_season', 'home_season_type',
    'home_TEAM_ABBREVIATION', 'away_TEAM_ABBREVIATION',

    # Cibles
    'home_win', 'point_diff',
    'home_PTS', 'away_PTS',

    # Stats basiques match
    'home_FG_PCT', 'away_FG_PCT',
    'home_FG3_PCT', 'away_FG3_PCT',
    'home_FT_PCT', 'away_FT_PCT',
    'home_REB', 'away_REB',
    'home_AST', 'away_AST',
    'home_TOV', 'away_TOV',
    'home_PLUS_MINUS', 'away_PLUS_MINUS',

    # Stats avancées match
    'adv_home_estimatedNetRating', 'adv_away_estimatedNetRating',
    'adv_home_estimatedOffensiveRating', 'adv_away_estimatedOffensiveRating',
    'adv_home_estimatedDefensiveRating', 'adv_away_estimatedDefensiveRating',
    'adv_home_estimatedPace', 'adv_away_estimatedPace',
    'adv_home_effectiveFieldGoalPercentage', 'adv_away_effectiveFieldGoalPercentage',
    'adv_home_trueShootingPercentage', 'adv_away_trueShootingPercentage',
    'adv_home_PIE', 'adv_away_PIE',
    'net_rating_diff', 'pace_diff', 'efg_diff',

    # Fatigue
    'home_rest_days', 'away_rest_days',
    'home_is_b2b', 'away_is_b2b',
    'rest_diff', 'b2b_advantage',

    # Forme récente (10 matchs)
    'home_roll_won_10', 'away_roll_won_10',
    'home_roll_pts_for_10', 'away_roll_pts_for_10',
    'home_roll_pts_against_10', 'away_roll_pts_against_10',
    'home_roll_pt_diff_10', 'away_roll_pt_diff_10',
    'win_pct_diff_10', 'pt_diff_diff_10',
]

# Garder uniquement les colonnes disponibles
available = [c for c in FEATURE_COLS if c in df.columns]
missing   = [c for c in FEATURE_COLS if c not in df.columns]

print(f'Features disponibles : {len(available)}')
if missing:
    print(f'Features manquantes  : {missing}')

Features disponibles : 57


In [17]:
df_ml = df[available].copy()

# Stats missing values
print('Missing values par colonne :')
missing_pct = (df_ml.isnull().sum() / len(df_ml) * 100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0].to_string())

Missing values par colonne :
win_pct_diff_10            1.194
pt_diff_diff_10            1.194
home_roll_pts_for_10       1.144
away_roll_pts_for_10       1.144
away_roll_won_10           1.144
home_roll_pts_against_10   1.144
away_roll_pts_against_10   1.144
home_roll_won_10           1.144
home_roll_pt_diff_10       1.144
away_roll_pt_diff_10       1.144
away_FT_PCT                0.025


In [18]:
# Supprimer les matchs sans stats avancées ni forme récente
# (premiers matchs de saison sans historique)
df_ml = df_ml.dropna(subset=['adv_home_estimatedNetRating', 'home_roll_won_10'])

print(f'DataFrame ML final : {df_ml.shape}')
print(f'Matchs avec toutes les features : {len(df_ml)}')
print(f'Taux victoire domicile : {df_ml["home_win"].mean():.1%}')
print(f'Point diff moyen : {df_ml["point_diff"].mean():.2f}')
df_ml.head(3)

DataFrame ML final : (3890, 57)
Matchs avec toutes les features : 3890
Taux victoire domicile : 55.8%
Point diff moyen : 2.29


,GAME_ID,date,home_season,home_season_type,home_TEAM_ABBREVIATION,away_TEAM_ABBREVIATION,home_win,point_diff,home_PTS,away_PTS,home_FG_PCT,away_FG_PCT,home_FG3_PCT,away_FG3_PCT,home_FT_PCT,away_FT_PCT,home_REB,away_REB,home_AST,away_AST,home_TOV,away_TOV,home_PLUS_MINUS,away_PLUS_MINUS,adv_home_estimatedNetRating,...,adv_home_effectiveFieldGoalPercentage,adv_away_effectiveFieldGoalPercentage,adv_home_trueShootingPercentage,adv_away_trueShootingPercentage,adv_home_PIE,adv_away_PIE,net_rating_diff,pace_diff,efg_diff,home_rest_days,away_rest_days,home_is_b2b,away_is_b2b,rest_diff,b2b_advantage,home_roll_won_10,away_roll_won_10,home_roll_pts_for_10,away_roll_pts_for_10,home_roll_pts_against_10,away_roll_pts_against_10,home_roll_pt_diff_10,away_roll_pt_diff_10,win_pct_diff_10,pt_diff_diff_10
43,22200049,2022-10-24,2022-23,Regular Season,MEM,BKN,1,10,134,124,0.500,0.540,0.471,0.310,0.774,0.808,38,35,21,27,9,11,10,-10,51.000,...,0.585,0.592,0.622,0.630,0.518,0.482,102.000,0.000,-0.007,1.000,2.000,0,0,-1.000,0,0.667,NaN,113.333,NaN,123.667,NaN,-10.333,NaN,NaN,NaN
44,22200044,2022-10-24,2022-23,Regular Season,PHI,IND,1,14,120,106,0.475,0.422,0.442,0.273,0.893,0.783,39,47,25,27,8,13,14,-14,81.300,...,0.594,0.489,0.650,0.529,0.583,0.417,162.600,0.000,0.105,1.000,1.000,0,0,0.000,0,0.000,0.333,103.333,121.667,110.000,122.000,-6.667,-0.333,-0.333,-6.333
46,22200050,2022-10-24,2022-23,Regular Season,MIN,SAS,0,-9,106,115,0.442,0.457,0.258,0.324,0.917,0.538,50,50,24,37,16,13,-9,9,-33.300,...,0.488,0.514,0.549,0.519,0.480,0.520,-66.600,0.000,-0.026,0.000,1.000,1,0,-1.000,-1,0.667,0.667,119.000,117.667,115.333,122.667,3.667,-5.000,0.000,8.667


In [19]:
# Sauvegarder
df_ml.to_csv('data/nba_processed.csv', index=False)
print(f'Sauvegardé : data/nba_processed.csv ({df_ml.shape})')

# Résumé final
print('\n=== RÉSUMÉ DATASET ML ===')
print(f'Matchs total          : {len(df_ml)}')
print(f'Features              : {len(df_ml.columns) - 5} (hors metadata)')
print(f'Saisons               : {df_ml["home_season"].unique()}')
print(f'Cible 1 — home_win    : binaire {df_ml["home_win"].value_counts().to_dict()}')
print(f'Cible 2 — point_diff  : min={df_ml["point_diff"].min():.0f} max={df_ml["point_diff"].max():.0f} moy={df_ml["point_diff"].mean():.1f}')

Sauvegardé : data/nba_processed.csv ((3890, 57))

=== RÉSUMÉ DATASET ML ===
Matchs total          : 3890
Features              : 52 (hors metadata)
Saisons               : <ArrowStringArray>
['2022-23', '2023-24', '2024-25']
Length: 3, dtype: str
Cible 1 — home_win    : binaire {1: 2172, 0: 1718}
Cible 2 — point_diff  : min=-56 max=62 moy=2.3
